# Student Interventions — Dataset Documentation

**Dataset:** `interventions.csv`

**Purpose:** Tracks student academic interventions, graduation support services, and intervention statuses within the synthetic Graduation Tracker student information system. Supports analysis of intervention frequency, service delivery, outstanding follow-up needs, and student support activities related to graduation readiness.

**Record level:** One row per student intervention event.

**Primary key:** `intervention_id`

**Foreign key:** `student_id` → `students.csv`

**Fields:**

- `intervention_id`: Unique intervention record identifier using the format `INT0000001`.
- `student_id`: Student identifier linking intervention records to the student dataset.
- `intervention_type`: Classification of the academic or graduation support intervention provided.
- `intervention_date`: Date of the intervention event, formatted as `YYYY-MM-DD`.
- `status`: Current intervention status (Open, Completed, or Follow-Up Needed).

**Business rules:**

- Each student receives between 0 and 4 randomly generated intervention records.
- Students may have no intervention records.
- Intervention types are randomly selected from six categories with equal probability:
  - Credit Recovery
  - Tutoring
  - Keystone Support
  - Graduation Planning
  - Family Outreach
  - Intervention Event
- Intervention dates are randomly generated within the 2026 calendar year.
- All intervention events are assigned to the 15th day of the randomly selected month.
- Intervention statuses follow these probability distributions:
  - Open: 25%
  - Completed: 55%
  - Follow-Up Needed: 20%
- Each intervention event receives one status.
- Multiple intervention records may exist for the same student.
- Multiple interventions of the same type may be assigned to the same student.
- Intervention IDs are sequentially generated and must be unique.
- Every `student_id` must exist in the student dataset.
- Random generation uses a fixed seed of 42 to support reproducibility.

**Data validation:**

- Verify all intervention IDs are unique.
- Verify all student IDs exist in the student dataset.
- Generate a count of intervention records by intervention type.

**Data source:** `students.csv` provides the student identifiers used to generate intervention records.

**Generation method:** Synthetic student intervention records generated using Python, pandas, and NumPy random sampling.

**Derived metrics:**

- `total_interventions`: Total number of intervention records associated with each student.
- `completed_interventions`: Number of intervention records with a Completed status.
- `open_interventions`: Number of intervention records with an Open status.
- `follow_up_needed`: Number of intervention records with a Follow-Up Needed status.
- `intervention_completion_rate`: Percentage of intervention records with a Completed status, calculated as completed interventions divided by total interventions multiplied by 100.

These metrics are not stored as separate fields in the exported dataset and must be calculated during subsequent analysis.

**Limitations:** Intervention types, dates, and statuses are synthetically generated and do not represent actual student support records. Intervention assignments are random and are not based on academic performance, Keystone assessment results, credit deficiencies, or graduation risk indicators. The dataset does not include intervention duration, assigned staff, student outcomes, follow-up deadlines, or measures of intervention effectiveness. Intervention statuses represent independently generated event classifications rather than a tracked intervention lifecycle. The dataset is intended for portfolio demonstration and does not establish causal relationships between interventions and student academic outcomes.

In [ ]:
# Import libraries

import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive

In [ ]:
# Connect Google Drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Create folder path

folder_path = Path('/content/drive/MyDrive/Graduation_Tracker/data/ground_truth')

In [ ]:
# Set random seed

rng = np.random.default_rng(42)

In [ ]:
# Generate interventions and follow-up events

students = pd.read_csv(folder_path/'students.csv')

records = []

for id in students.student_id:
  for r in range(int(rng.integers(0,5))):
    records.append({
        'student_id':id,
        'intervention_type':rng.choice([
            'Credit Recovery',
            'Tutoring',
            'Keystone Support',
            'Graduation Planning',
            'Family Outreach',
            'Intervention Event'
        ]),
        'intervention_date':f'2026-{int(rng.integers(1,13)):02d}-15',
        'status':rng.choice(['Open', 'Completed', 'Follow-Up Needed'], p=[0.25, 0.55, 0.20])
    })

interventions = pd.DataFrame(records)

interventions.insert(0, 'intervention_id', [f'INT{i:07d}' for i in range(1, len(interventions)+1)])

display(interventions.head())

,intervention_id,student_id,intervention_type,intervention_date,status
0,INT0000001,1145270,Graduation Planning,2026-06-15,Follow-Up Needed
1,INT0000002,1145270,Credit Recovery,2026-09-15,Open
2,INT0000003,1145270,Graduation Planning,2026-12-15,Completed
3,INT0000004,1145271,Family Outreach,2026-07-15,Completed
4,INT0000005,1145271,Credit Recovery,2026-07-15,Follow-Up Needed


In [ ]:
# Validate dataset

assert interventions.intervention_id.is_unique

assert interventions.student_id.isin(students.student_id).all()

print(interventions.groupby('intervention_type').size())

intervention_type
Credit Recovery        1656
Family Outreach        1651
Graduation Planning    1643
Intervention Event     1618
Keystone Support       1679
Tutoring               1632
dtype: int64


In [ ]:
# Save CSV to Google Drive

csv_path = folder_path/'interventions.csv'

interventions.to_csv(csv_path, index = False)